# 环节 11 · 平台化落地演示

纯 Python 标准库，零依赖。手搓四件事：

1. Docker 加固配置校验器；
2. K8s `securityContext` 对 `restricted` 基线的校验；
3. 多租户配额、超配率与 P99 排队；
4. 三种计费口径的对比。

> 把「加固」从文档变成可校验的配置断言。

In [ ]:
# §1 Docker 加固配置校验器
REQUIRED = {
    "cap_drop_all":      ("--cap-drop ALL", "容器内可能持有 CAP_SYS_ADMIN"),
    "no_new_privileges": ("--security-opt no-new-privileges", "setuid 可提权"),
    "read_only_rootfs":  ("--read-only", "可改镜像内文件（含提权路径）"),
    "non_root_user":     ("--user <非 0>", "容器内 root ≈ 宿主高权"),
    "pids_limit":        ("--pids-limit", "fork 炸弹"),
    "memory_limit":      ("--memory + --memory-swap 同值", "可用 swap 绕开内存限制"),
    "network_isolated":  ("--network none", "直接出网外传"),
    "seccomp_profile":   ("--security-opt seccomp=...", "内核攻击面全开"),
    "input_readonly":    ("type=bind,...,readonly", "输入被沙箱改坏"),
    "no_docker_sock":    ("不挂 /var/run/docker.sock", "一步逃逸"),
}

GOOD = {k: True for k in REQUIRED}
BAD = dict(GOOD, network_isolated=False, no_docker_sock=False, read_only_rootfs=False)


def audit_docker(cfg):
    gaps = [(k, REQUIRED[k][1]) for k in REQUIRED if not cfg.get(k)]
    return gaps


for label, cfg in [("加固后", GOOD), ("图省事", BAD)]:
    gaps = audit_docker(cfg)
    print(f"{label}: 通过 {len(REQUIRED) - len(gaps)}/{len(REQUIRED)}")
    for k, why in gaps:
        print(f"  ✗ {k:<20} 后果：{why}")
    print()

In [ ]:
# §2 K8s securityContext 对 restricted 基线的校验
K8S_CHECKS = {
    "runAsNonRoot":            lambda c: c.get("runAsNonRoot") is True,
    "allowPrivilegeEscalation": lambda c: c.get("allowPrivilegeEscalation") is False,
    "privileged":              lambda c: c.get("privileged") is False,
    "readOnlyRootFilesystem":  lambda c: c.get("readOnlyRootFilesystem") is True,
    "capabilities.drop ALL":   lambda c: "ALL" in (c.get("capabilities", {}).get("drop") or []),
    "seccompProfile":          lambda c: c.get("seccompProfile", {}).get("type") in ("RuntimeDefault", "Localhost"),
    "no hostPath/hostNetwork": lambda c: not c.get("hostPath") and not c.get("hostNetwork"),
    "automountSAToken=false":  lambda c: c.get("automountServiceAccountToken") is False,
    "ephemeral-storage 限额":  lambda c: bool(c.get("resources", {}).get("limits", {}).get("ephemeral-storage")),
}

GOOD_POD = {
    "runAsNonRoot": True, "allowPrivilegeEscalation": False, "privileged": False,
    "readOnlyRootFilesystem": True, "capabilities": {"drop": ["ALL"]},
    "seccompProfile": {"type": "RuntimeDefault"},
    "hostPath": None, "hostNetwork": False, "automountServiceAccountToken": False,
    "resources": {"limits": {"ephemeral-storage": "1Gi"}},
}
BAD_POD = dict(GOOD_POD, readOnlyRootFilesystem=False, allowPrivilegeEscalation=True,
               automountServiceAccountToken=True)
BAD_POD.pop("resources")

for label, pod in [("restricted 合规", GOOD_POD), ("常见漏配", BAD_POD)]:
    failed = [k for k, f in K8S_CHECKS.items() if not f(pod)]
    print(f"{label}: {len(K8S_CHECKS) - len(failed)}/{len(K8S_CHECKS)} 通过")
    for k in failed:
        print(f"  ✗ {k}")
    print()
print("兜底：命名空间打 pod-security.kubernetes.io/enforce=restricted，")
print("      即使某个团队忘写 securityContext 也会被拦下")

In [ ]:
# §3 多租户配额、超配率与 P99 排队（近似模型）
CLUSTER_CORES = 100
PER_TASK_CORES = 1
BASE_SERVICE_S = 2.0     # 无竞争时的服务时长


def row(oversub, n_tasks=200):
    allowed_conc = CLUSTER_CORES * oversub / PER_TASK_CORES
    nominal_conc = CLUSTER_CORES / PER_TASK_CORES
    # 近似：并发超过标称容量后，每个任务分到的算力按比例下降
    slowdown = max(1.0, allowed_conc / nominal_conc)
    service = BASE_SERVICE_S * slowdown
    # 利用率近似（越高排队越久），M/M/1 近似 Wq ≈ ρ/(1-ρ)·service
    rho = min(0.98, 0.6 + 0.35 * (oversub - 1))
    wq = rho / (1 - rho) * service
    return allowed_conc, service, wq


print(f"{'超配率':<8}{'允许并发':<10}{'单任务服务(s)':<14}{'排队 P50≈Wq(s)'}")
print("-" * 52)
for oversub in [1.0, 1.5, 2.0, 3.0]:
    conc, svc, wq = row(oversub)
    print(f"{oversub:<8}{conc:<10.0f}{svc:<14.2f}{wq:.2f}")
print()
print("结论：超配省钱，但排队时间非线性上升 —— 容量规划必须看 P99 而非均值")
print("（真实平台还要叠加：调度延迟、镜像拉取、节点级资源竞争）")

In [ ]:
# §4 三种计费口径对同一批任务的差异
def bill_by_time(durations_h, price_per_hour=0.5):
    return sum(durations_h) * price_per_hour


def bill_by_resource(durations_h, vcpu=1.0, mem_gb=0.5,
                     p_cpu=0.4, p_mem=0.05):
    return sum(d * (vcpu * p_cpu + mem_gb * p_mem) for d in durations_h)


def bill_by_invocation(n, price_per_call=0.002):
    return n * price_per_call


TASKS_H = [0.05] * 100 + [0.01] * 100     # 100 个 3 分钟任务 + 100 个 36 秒任务
N = len(TASKS_H)

a = bill_by_time(TASKS_H)
b = bill_by_resource(TASKS_H)
c = bill_by_invocation(N)

print(f"任务数 {N}，总时长 {sum(TASKS_H) * 60:.0f} 分钟")
print(f"按时长计费      : {a:.3f}")
print(f"按资源用量计费  : {b:.3f}")
print(f"按调用数计费    : {c:.3f}")
print()
print("口径影响行为：")
print("  - 按调用 → 激励『把多个小任务合并进一个沙箱』（减少创建次数）")
print("  - 按时长 → 预热池空转也要收钱（需说明规则）")
print("  - 按资源 → 最公平，但**依赖用量采集**（即环节 09 的事件流）")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | K8s 里怎么提高隔离档？ | `runtimeClassName`（gVisor / Kata）——一行切换 |
| 2 | 为什么要开 `restricted` 兜底？ | 逐 Pod 声明会漏；命名空间级兜底能挡住「忘记加固」 |
| 3 | `requests` 与 `limits` 都要设？ | 按 requests 调度、按 limits 限流；缺一会调度失真或互相抢占 |
| 4 | 超配率高的代价？ | P99 排队非线性上升；均值掩盖问题 |
| 5 | 多租户「分 namespace」够吗？ | 不够；共享内核抹不掉，要靠 RuntimeClass / 节点池 / 独立集群 |
| 6 | 计费为什么依赖可观测性？ | 用量从审计事件采集；口径还决定用户行为 |

**相关长文**：[环节11-平台化落地详解.md](./环节11-平台化落地详解.md)